In [ ]:
"""
=============================================================
FILE 33 — MAP REDUCE PATTERN
=============================================================

CONCEPTS TAUGHT
----------------
1. Map Reduce Architecture
2. Large Scale Processing
3. Parallel Chunk Processing
4. Aggregation
5. Distributed AI Workflows
6. Document Summarization
7. Scalable Processing
8. Batch AI Systems
9. Parallel Reasoning
10. Enterprise Data Processing

CORE IDEA
-----------
Split a large problem into chunks,
process independently,
then combine results.

FLOW
-----
Large Input
   ↓
Chunk Split
   ↓
Parallel Processing
   ↓
Aggregation
   ↓
Final Output

REAL WORLD USE CASES
---------------------
- Long document summarization
- Enterprise analytics
- Research pipelines
- Large-scale RAG
"""

# ============================================================
# STEP 1 — IMPORTS
# ============================================================

import os

from dotenv import load_dotenv

from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END

from IPython.display import Image, display

# ============================================================
# STEP 2 — LOAD ENV VARIABLES
# ============================================================

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# ============================================================
# STEP 3 — INITIALIZE LLM
# ============================================================

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

# ============================================================
# STEP 4 — DEFINE STATE
# ============================================================

class State(TypedDict):
    document: str
    summary_part_1: str
    summary_part_2: str
    summary_part_3: str
    final_summary: str

# ============================================================
# STEP 5 — MAP NODES
# ============================================================

def summarize_part_1(state: State):

    chunk = state["document"][:500]

    response = llm.invoke(
        f"Summarize:\n{chunk}"
    )

    return {
        "summary_part_1": response.content
    }

def summarize_part_2(state: State):

    chunk = state["document"][500:1000]

    response = llm.invoke(
        f"Summarize:\n{chunk}"
    )

    return {
        "summary_part_2": response.content
    }

def summarize_part_3(state: State):

    chunk = state["document"][1000:1500]

    response = llm.invoke(
        f"Summarize:\n{chunk}"
    )

    return {
        "summary_part_3": response.content
    }

# ============================================================
# STEP 6 — REDUCE NODE
# ============================================================

def aggregate_summary(state: State):

    response = llm.invoke(
        f"""
        Combine these summaries into one final summary.

        PART 1:
        {state['summary_part_1']}

        PART 2:
        {state['summary_part_2']}

        PART 3:
        {state['summary_part_3']}
        """
    )

    return {
        "final_summary": response.content
    }

# ============================================================
# STEP 7 — BUILD GRAPH
# ============================================================

builder = StateGraph(State)

builder.add_node("summarize_part_1", summarize_part_1)
builder.add_node("summarize_part_2", summarize_part_2)
builder.add_node("summarize_part_3", summarize_part_3)

builder.add_node("aggregate_summary", aggregate_summary)

# ============================================================
# STEP 8 — PARALLEL EXECUTION
# ============================================================

builder.add_edge(START, "summarize_part_1")
builder.add_edge(START, "summarize_part_2")
builder.add_edge(START, "summarize_part_3")

builder.add_edge("summarize_part_1", "aggregate_summary")
builder.add_edge("summarize_part_2", "aggregate_summary")
builder.add_edge("summarize_part_3", "aggregate_summary")

builder.add_edge("aggregate_summary", END)

# ============================================================
# STEP 9 — COMPILE GRAPH
# ============================================================

graph = builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))

# ============================================================
# STEP 10 — SAMPLE DOCUMENT
# ============================================================

document = """
Artificial Intelligence is transforming industries globally.
Retail companies are using AI for recommendation systems.
Healthcare uses AI for diagnosis and prediction.
Finance uses AI for fraud detection and automation.
Education uses AI for personalized learning.
""" * 20

# ============================================================
# STEP 11 — RUN WORKFLOW
# ============================================================

result = graph.invoke(
    {
        "document": document
    }
)

# ============================================================
# STEP 12 — PRINT RESULT
# ============================================================

print("\nFINAL SUMMARY\n")
print("=" * 60)
print(result["final_summary"])